# Q2: Data Cleaning

**Phase 3:** Data Cleaning & Preprocessing  
**Points: 9 points**

**Focus:** Handle missing data, outliers, validate data types, remove duplicates.

**Lecture Reference:** Lecture 11, Notebook 1 ([`11/demo/01_setup_exploration_cleaning.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/01_setup_exploration_cleaning.ipynb)), Phase 3. Also see Lecture 05 (data cleaning).

---

## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load data from Q1 (or directly from source)
df = pd.read_csv('data/beach_sensors.csv')
df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
df['Measurement Timestamp Label'] = pd.to_datetime(df['Measurement Timestamp Label'])
# If you saved cleaned data from Q1, you can load it:
# df = pd.read_csv('output/q1_exploration.csv')  # This won't work - load original

## Cleaning data/beach_sensors.csv

In [2]:
# 1. `output/q2_cleaned_data.csv`

# Begin Chunk
print('Begin Cleaning Process of Beach Sensors dataset ...')

# See if data is missing by year
df['Year'] = df['Measurement Timestamp'].dt.year
missing_year = df.groupby('Year').apply(lambda x: x.isnull().sum())
print(missing_year) ## Randomly Spread by Year

# See if data is missing by station
missing_station = df.groupby('Station Name').apply(lambda x: x.isnull().sum())
print(missing_station) ## Foster Weather Station Primary Source Missing Data

# See Number of Rows From Foster Weather Station
foster_rows = len(df[df['Station Name'] == 'Foster Weather Station'])
print(foster_rows) ## Missingness Caused by FWS Not Recording This Data

# For Valid Modelling Inference, Remove Rows Corresponding to FWS
df_clean = df[df['Station Name'] != 'Foster Weather Station']

# Barometric Pressure 73 Missing | Air Temperature 75 Missing
print(df_clean.isnull().sum())  

# Check for Duplicates
print(f'Number of duplicates: {df_clean.duplicated().sum()}') # 0

# Outlier Function | From Demo
def detect_outliers_iqr(df, column, iqr_multiplier=1.5):
    Q1 = df[column].quantile(0.25)  # 25th percentile
    Q3 = df[column].quantile(0.75)  # 75th percentile
    IQR = Q3 - Q1  # Interquartile range (middle 50% of data)
    # Tukey fences: standard statistical method for outlier detection
    lower_bound = Q1 - iqr_multiplier * IQR
    upper_bound = Q3 + iqr_multiplier * IQR
    # Find values outside the fences
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Identify Unusual Values
numeric_columns = df_clean.select_dtypes(include = 'number').columns
for column in numeric_columns:
    if df_clean[column].notna().sum() > 0:
        outliers, lower, upper = detect_outliers_iqr(df_clean, column, iqr_multiplier = 2)
        if len(outliers) > 0:
            print(f'\n {column}: Lower: {lower:.1f}, Upper: {upper:.1f}')

# Apply Domain Knowledge Caps
df_clean['Wind Speed'] = df_clean['Wind Speed'].clip(lower = 0, upper = 87) # Highest Windspeed Chicago Ever
df_clean['Maximum Wind Speed'] = df_clean['Maximum Wind Speed'].clip(lower = 0, upper = 87) 

# Convert Unusual Values to NaN for Imputation
df_clean.loc[df_clean['Total Rain'] < 0, 'Total Rain'] = np.nan
df_clean.loc[df_clean['Solar Radiation'] < 0, 'Solar Radiation'] = np.nan
df_clean.loc[df_clean['Barometric Pressure'] == 0, 'Barometric Pressure'] = np.nan
df_clean.loc[df_clean['Maximum Wind Speed'] == 0, 'Maximum Wind Speed'] = np.nan


# Forward Fill Impute NaN
nan_columns = df_clean.columns[df_clean.isnull().any()]
df_clean = df_clean.sort_values(['Station Name', 'Measurement Timestamp'])
for column in nan_columns:
    df_clean[column] = df_clean.groupby('Station Name')[column].fillna(method = 'ffill')


# Check Imputation Works
print(f'Any NaN remaining:\n{df_clean.isnull().sum()}') # Solar Radiation NaN

# Address Solar Radiation
print(df_clean[df_clean['Solar Radiation'].isna()][['Station Name', 'Measurement Timestamp', 'Solar Radiation']].head(10)) # Night Hours => Set to 0
df_clean['Solar Radiation'] = df_clean['Solar Radiation'].fillna(0)

# Drop First 75 Rows to Account for Non-Recordings of First 75 Air Temperatures and 73 Barometric Pressures
df_clean = df_clean.dropna()

# Verify Cleaning
print(f'Any NaN remaining:\n{df_clean.isnull().sum()}')

# Verify Not Overdropping
print(f'Dataset Shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')

# Output q2_cleaned_data.csv
df_clean = df_clean.drop(columns = ['Year'])
df_clean.to_csv('output/q2_cleaned_data.csv', index = False)

# End chunk
print('Cleaning Process has successfully been completed. Clean data stored at: output/q2_cleaned_data.csv')

Begin Cleaning Process of Beach Sensors dataset ...
      Station Name  Measurement Timestamp  Air Temperature  \
Year                                                         
2015             0                      0               75   
2016             0                      0                0   
2017             0                      0                0   
2018             0                      0                0   
2019             0                      0                0   
2020             0                      0                0   
2021             0                      0                0   
2022             0                      0                0   
2023             0                      0                0   
2024             0                      0                0   
2025             0                      0                0   

      Wet Bulb Temperature  Humidity  Rain Intensity  Interval Rain  \
Year                                                                  

/var/folders/q2/425x2zhd1b3f3hnjp4v223880000gn/T/ipykernel_91434/2414268887.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_year = df.groupby('Year').apply(lambda x: x.isnull().sum())
/var/folders/q2/425x2zhd1b3f3hnjp4v223880000gn/T/ipykernel_91434/2414268887.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_station = df.groupby('Station Name').apply(lambda x: x.isnull().sum())



 Rain Intensity: Lower: 0.0, Upper: 0.0

 Interval Rain: Lower: 0.0, Upper: 0.0

 Total Rain: Lower: -350.9, Upper: 568.1

 Precipitation Type: Lower: 0.0, Upper: 0.0

 Wind Speed: Lower: -3.3, Upper: 8.2

 Maximum Wind Speed: Lower: -4.7, Upper: 13.3

 Barometric Pressure: Lower: 973.8, Upper: 1015.8

 Solar Radiation: Lower: -343.0, Upper: 517.0

 Heading: Lower: 336.0, Upper: 371.0

 Battery Life: Lower: 11.7, Upper: 12.2
Any NaN remaining:
Station Name                    0
Measurement Timestamp           0
Air Temperature                75
Wet Bulb Temperature            0
Humidity                        0
Rain Intensity                  0
Interval Rain                   0
Total Rain                      0
Precipitation Type              0
Wind Direction                  0
Wind Speed                      0
Maximum Wind Speed              0
Barometric Pressure            73
Solar Radiation                 0
Heading                         0
Battery Life                    0
Measure

/var/folders/q2/425x2zhd1b3f3hnjp4v223880000gn/T/ipykernel_91434/2414268887.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Wind Speed'] = df_clean['Wind Speed'].clip(lower = 0, upper = 87) # Highest Windspeed Chicago Ever
/var/folders/q2/425x2zhd1b3f3hnjp4v223880000gn/T/ipykernel_91434/2414268887.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Maximum Wind Speed'] = df_clean['Maximum Wind Speed'].clip(lower = 0, upper = 87)
/var/folders/q2/425x2zhd1b3f3hnjp4v223880000gn/T/ipyk

Cleaning Process has successfully been completed. Clean data stored at: output/q2_cleaned_data.csv


## Report on Cleaning Process

In [3]:
# 2. `output/q2_cleaning_report.txt`

# Begin chunk
print('Beginning generation of cleaning report text file ...')

# Write Cleaning Report
with open('output/q2_cleaning_report.txt', 'w') as f:
    f.write('DATA CLEANING REPORT\n')
    f.write('====================\n\n')

    f.write(f'Rows before cleaning: {len(df)}\n\n')

    f.write('Missing Data Handling:\n')
    f.write(f'- Foster Weather Station: {foster_rows} rows (38.7%)\n')
    f.write('Method: Removed station from dataset\n')
    f.write('Result: All rows removed from dataset\n\n')

    f.write('- Air Temperature: 75 missing values (0.0%)\n')
    f.write('Method: Forward-fill imputation by station\n')
    f.write('Result: All missing values filled\n\n')

    f.write('- Barometric Pressure: 73 missing values + 6 improperly coded (0.0%)\n')
    f.write('Method: Improperly coded values converted to NaN. Forward-fill imputation by station\n')
    f.write('Result: All missing values filled\n\n')

    f.write('- Total Rain: Negative values present\n')
    f.write('Method: Negative values converted to NaN. Forward-fill imputation by station\n')
    f.write('Result: All missing values filled\n\n')

    f.write('- Solar Radiation: Negative values and nighttime missing data\n')
    f.write('Method: Negative values set to NaN, forward-fill. Remaining missing value (nighttime) set to 0\n')
    f.write('Result: All missing values filled\n\n')

    f.write('- Maximum Wind Speed: Zero values detected\n')
    f.write('Method: Zero values set to NaN, forward-fill imputation\n')
    f.write('Result: All missing values filled\n\n')

    f.write('Outlier Handling:\n')
    f.write('Method: IQR Method (2xIQR) at Domain Knowledge\n')
    f.write('- Wind Speed + Maximum Wind Speed: Capped [0, 87]\n')
    f.write('Result: Negative values and values greater than ever recorded values set to bounds\n\n')

    f.write('Duplicates Removed: 0\n\n')

    f.write('Data Type Conversions:\n')
    f.write('- Measurement Timestamp: Converted to datetime64[ns]\n')
    f.write('- Measurement Timestamp Label: Converted to datetime64[ns]\n\n')

    f.write(f'Rows after cleaning: {len(df_clean)}')

# End chunk
print('Cleaning report text file has been successfully generated at: output/q2_cleaning_report.txt')

Beginning generation of cleaning report text file ...
Cleaning report text file has been successfully generated at: output/q2_cleaning_report.txt


## Number of Rows Remaining in Cleaned Dataset

In [4]:
# 3. `output/q2_rows_cleaned.txt`

# Begin chunk
print('Generating text file showing the number of rows in the cleaned dataset ...')

# Create txt file
with open('output/q2_rows_cleaned.txt', 'w') as f:
    f.write(str(df_clean.shape[0]))

# End chunk
print('Number of rows have been successfully calculated and stored at: output/q2_rows_cleaned.txt')

Generating text file showing the number of rows in the cleaned dataset ...
Number of rows have been successfully calculated and stored at: output/q2_rows_cleaned.txt


---

## Objective

Clean the dataset by handling missing data, outliers, validating data types, and removing duplicates.

**Time Series Note:** For time series data, forward-fill (`ffill()`) is often appropriate for missing values since sensor readings are continuous. However, you may choose other strategies based on your analysis.

---

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q2_cleaned_data.csv`
**Format:** CSV file
**Content:** Cleaned dataset with same structure as original (same columns)
**Requirements:**
- Same columns as original dataset
- Missing values handled (filled, dropped, or imputed)
- Outliers handled (removed, capped, or transformed)
- Data types validated and converted
- Duplicates removed
- **Sanity check:** Dataset should retain most rows after cleaning (at least 1,000 rows). If you're removing more than 50% of data, reconsider your strategy—imputation is usually preferable to dropping rows for this dataset.
- **No index column** (save with `index=False`)

### 2. `output/q2_cleaning_report.txt`
**Format:** Plain text file
**Content:** Detailed report of cleaning operations
**Required information:**
- Rows before cleaning: [number]
- Missing data handling method: [description]
  - Which columns had missing data
  - Method used (drop, forward-fill, impute, etc.)
  - Number of values handled
- Outlier handling: [description]
  - Detection method (IQR, z-scores, domain knowledge)
  - Which columns had outliers
  - Method used (remove, cap, transform)
  - Number of outliers handled
- Duplicates removed: [number]
- Data type conversions: [list any conversions]
- Rows after cleaning: [number]

**Example format:**
```
DATA CLEANING REPORT
====================

Rows before cleaning: 50000

Missing Data Handling:
- Water Temperature: 2500 missing values (5.0%)
  Method: Forward-fill (time series appropriate)
  Result: All missing values filled
  
- Air Temperature: 1500 missing values (3.0%)
  Method: Forward-fill, then median imputation for remaining
  Result: All missing values filled

Outlier Handling:
- Water Temperature: Detected 500 outliers using IQR method (3×IQR)
  Method: Capped at bounds [Q1 - 3×IQR, Q3 + 3×IQR]
  Bounds: [-5.2, 35.8]
  Result: 500 values capped

Duplicates Removed: 0

Data Type Conversions:
- Measurement Timestamp: Converted to datetime64[ns]

Rows after cleaning: 50000
```

### 3. `output/q2_rows_cleaned.txt`
**Format:** Plain text file
**Content:** Single integer number (total rows after cleaning)
**Requirements:**
- Only the number, no text, no labels
- No whitespace before or after
- Example: `50000`

---

## Requirements Checklist

- [ ] Missing data handling strategy chosen and implemented
- [ ] Outliers detected and handled (IQR method, z-scores, or domain knowledge)
- [ ] Data types validated and converted
- [ ] Duplicates identified and removed
- [ ] Cleaning decisions documented in report
- [ ] All 3 required artifacts saved with exact filenames

---

## Your Approach

1. **Handle missing data** - Choose appropriate strategy (drop, forward-fill, impute) based on data characteristics
2. **Detect and handle outliers** - Use IQR method or z-scores; decide whether to remove, cap, or transform
3. **Validate data types** - Ensure numeric and datetime columns are properly typed
4. **Remove duplicates**
5. **Document and save** - Write detailed cleaning report explaining your decisions

---

## Decision Points

- **Missing data:** Should you drop rows, impute values, or forward-fill? Consider: How much data is missing? Is it random or systematic? For time series, forward-fill is often appropriate.
- **Outliers:** Are they errors or valid extreme values? Use IQR method or z-scores to detect, then decide: remove, cap, or transform. Document your reasoning.
- **Data types:** Are numeric columns actually numeric? Are datetime columns properly formatted? Convert as needed.

---

## Checkpoint

After Q2, you should have:
- [ ] Missing data handled
- [ ] Outliers addressed
- [ ] Data types validated
- [ ] Duplicates removed
- [ ] All 3 artifacts saved: `q2_cleaned_data.csv`, `q2_cleaning_report.txt`, `q2_rows_cleaned.txt`

---

**Next:** Continue to `q3_data_wrangling.md` for Data Wrangling.
